In [ ]:
# Message flow
#
# Get trace data using the provided launch file:
#    $ ros2 launch tracetools_analysis message_flow.launch.py
#    (wait at least a few seconds, then kill with Ctrl+C)
#
# OR
#
# Use the provided sample converted trace file, changing the path below to:
#    'sample_data/converted_message_flow'

In [ ]:
path = '~/.ros/tracing/message_flow/ust'
#path = 'sample_data/converted_message_flow'

In [ ]:
# Set begin and end topics to analyze the message flow between them.
# The script will process any messages that are published on the end topic and
# backtrace the flow until it finds a message published on one of the provided begin topics.
# The flow of messages between these topics will be analyzed and visualized.
# If begin_topic is a list, the earliest occurrence of any of the topics in the list will be used as the starting point for analysis.

begin_topic = [
    "/ping",
    "/pong",
]
end_topic = "/combined"

max_num_analyze = None  # Set to None to analyze all

In [ ]:
import sys
# Add paths to tracetools_analysis and tracetools_read.
# There are two options:
#   1. from source, assuming a workspace with:
#       src/tracetools_analysis/
#       src/ros2/ros2_tracing/tracetools_read/
sys.path.insert(0, '../')
sys.path.insert(0, '../../../ros2/ros2_tracing/tracetools_read/')
sys.path.insert(0, '../../../ros2/ros2_tracing/tracetools_trace/')
#   2. from Debian packages, setting the right ROS 2 distro:
#ROS_DISTRO = 'rolling'
#sys.path.insert(0, f'/opt/ros/{ROS_DISTRO}/lib/python3.8/site-packages')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tracetools_analysis.loading import load_file
from tracetools_analysis.processor.ros2 import Ros2Handler

In [ ]:
# Load and process trace data
events = load_file(path)
handler = Ros2Handler.process(events)

# Map all publisher and subscription handles to their topics
rcl_publisher_to_topic_map = {}
for rcl_publisher, row in handler.data.rcl_publishers.iterrows():
    rcl_publisher_to_topic_map[rcl_publisher] = row['topic_name']
rcl_subscription_to_topic_map = {}
for rcl_subscription, row in handler.data.rcl_subscriptions.iterrows():
    rcl_subscription_to_topic_map[rcl_subscription] = row['topic_name']

# Map all rcl publisher and subscription handles to their rmw handles
rcl_publisher_to_rmw_map = {}
for rcl_publisher, row in handler.data.rcl_publishers.iterrows():
    rcl_publisher_to_rmw_map[rcl_publisher] = row['rmw_handle']
rcl_subscription_to_rmw_map = {}
for rcl_subscription, row in handler.data.rcl_subscriptions.iterrows():
    rcl_subscription_to_rmw_map[rcl_subscription] = row['rmw_handle']

# Map all rmw publisher and subscription handles to their topics
rmw_publisher_to_topic_map = {}
for rcl_publisher, topic in rcl_publisher_to_topic_map.items():
    rmw_handle = rcl_publisher_to_rmw_map[rcl_publisher]
    rmw_publisher_to_topic_map[rmw_handle] = topic
rmw_subscription_to_topic_map = {}
for rcl_subscription, topic in rcl_subscription_to_topic_map.items():
    rmw_handle = rcl_subscription_to_rmw_map[rcl_subscription]
    rmw_subscription_to_topic_map[rmw_handle] = topic

# Map all rcl publishers to their subscriptions via annotated message links
rcl_periodic_publisher_to_subscriptions_map = {}
for _, row in handler.data.message_links_periodic_async.iterrows():
    for rcl_publisher in row['pubs']:
        if rcl_publisher in rcl_periodic_publisher_to_subscriptions_map:
            print(f"Warning: Overwriting more recent periodic message link for rcl_publisher: {rcl_publisher}")
        rcl_periodic_publisher_to_subscriptions_map[rcl_publisher] = set()
        for rcl_subscription in row['subs']:
            rcl_periodic_publisher_to_subscriptions_map[rcl_publisher].add(rcl_subscription)
rcl_partial_publisher_to_subscriptions_map = {}
for _, row in handler.data.message_links_partial_sync.iterrows():
    for rcl_publisher in row['pubs']:
        if rcl_publisher in rcl_partial_publisher_to_subscriptions_map:
            print(f"Warning: Overwriting more recent partial message link for rcl_publisher: {rcl_publisher}")
        rcl_partial_publisher_to_subscriptions_map[rcl_publisher] = set()
        for rcl_subscription in row['subs']:
            rcl_partial_publisher_to_subscriptions_map[rcl_publisher].add(rcl_subscription)

In [ ]:
def timestamp_to_readable(timestamp_ns):
    """Convert nanoseconds since epoch to HH:MM:SS.nnnnnnnnn format"""
    dt = pd.to_datetime(timestamp_ns, unit='ns', utc=True)
    # Get the time part and the nanosecond fraction
    time_str = dt.strftime('%H:%M:%S')
    nanoseconds = timestamp_ns % 1_000_000_000
    return f"{time_str}.{nanoseconds:09d}"

def get_dependencies_of_publisher(rcl_publish_instance):
    rcl_publisher = rcl_publish_instance.publisher_handle
    
    # Option 1: Message is published periodically depending on specified subscriptions (annotated with TRACETOOLS_TRACEPOINT() in source code)
    if rcl_publisher in rcl_periodic_publisher_to_subscriptions_map:
        subscriptions = rcl_periodic_publisher_to_subscriptions_map[rcl_publisher]
        print(f"    {rcl_publisher_to_topic_map[rcl_publisher]} <-- periodic async <-- {[rcl_subscription_to_topic_map[sub] for sub in subscriptions]}")
    
    # Option 2: Message is published when all specified subscriptions have taken messages (annotated with TRACETOOLS_TRACEPOINT() in source code)
    elif rcl_publisher in rcl_partial_publisher_to_subscriptions_map:
        subscriptions = rcl_partial_publisher_to_subscriptions_map[rcl_publisher]
        print(f"    {rcl_publisher_to_topic_map[rcl_publisher]} <-- partial sync <-- {[rcl_subscription_to_topic_map[sub] for sub in subscriptions]}")

    # Option 3: No annotated message links found for this publisher. Check if message is published during a subscription callback.    
    else:
        # check if published during a subscription callback
        rcl_callbacks_parallel = handler.data.callback_instances[
            (handler.data.callback_instances.timestamp.astype(np.int64) <= rcl_publish_instance.timestamp) &
            (handler.data.callback_instances.timestamp.astype(np.int64) + handler.data.callback_instances.duration.astype(np.int64) >= rcl_publish_instance.timestamp) &
            (handler.data.callback_instances.pid == rcl_publish_instance.pid) &
            (handler.data.callback_instances.tid == rcl_publish_instance.tid) &
            (handler.data.callback_instances.procname == rcl_publish_instance.procname)
        ]

        if rcl_callbacks_parallel.empty:
            raise ValueError(f"    {rcl_publisher_to_topic_map[rcl_publisher]} <-- no dependencies found")
        elif len(rcl_callbacks_parallel) > 1:
            raise ValueError(f"❌ Could not identify single callback for rcl_publisher 0x{rcl_publisher:x} at timestamp {rcl_publish_instance.timestamp}")
        else:
            rcl_callback_object = handler.data.callback_objects[handler.data.callback_objects.callback_object == rcl_callbacks_parallel.iloc[0].callback_object]
            rcl_sub_handles = handler.data.subscription_objects[handler.data.subscription_objects.index == rcl_callback_object.index[0]].subscription_handle
            subscriptions = None
            for rcl_sub_handle in rcl_sub_handles:
                if rcl_subscription_to_topic_map[rcl_sub_handle] != '/clock':
                    subscriptions = {rcl_sub_handle}
            if not subscriptions:
                raise ValueError(f"❌ Could not identify subscription for callback object {rcl_callback_object.iloc[0].callback_object}")
            print(f"    {rcl_publisher_to_topic_map[rcl_publisher]} <-- callback <-- {[rcl_subscription_to_topic_map[sub] for sub in subscriptions]}")
    
    return subscriptions

def get_latest_taken_message_for_subscription(rcl_subscription, timestamp):
    taken_messages = handler.data.rmw_take_instances[
        (handler.data.rmw_take_instances['subscription_handle'] == rcl_subscription_to_rmw_map[rcl_subscription]) &
        (handler.data.rmw_take_instances['timestamp'] <= timestamp)
    ]
    if taken_messages.empty:
        return None
    latest_taken_message = taken_messages.loc[taken_messages['timestamp'].idxmax()]
    return latest_taken_message

def get_published_for_taken_message(rmw_take_instance):
    taken_topic = rmw_subscription_to_topic_map[rmw_take_instance.subscription_handle]
    topic_publishers = [rmw_publisher for rmw_publisher, topic in rmw_publisher_to_topic_map.items() if topic == taken_topic]
    published_messages = handler.data.rmw_publish_instances[
        (handler.data.rmw_publish_instances['publisher_handle'].isin(topic_publishers)) &
        (handler.data.rmw_publish_instances['timestamp'] == rmw_take_instance.source_timestamp)
    ]
    if published_messages.empty:
        return None
    if len(published_messages) > 1:
        raise ValueError("❌ Could not identify single published for taken message")
    return published_messages.iloc[0]

def backpropagate_messages(rcl_publish_instance_end, begin_topics, time):
    if isinstance(begin_topics, str):
        begin_topics = [begin_topics]
    
    rcl_publish_topic = rcl_publisher_to_topic_map[rcl_publish_instance_end.publisher_handle]
    print(f"  ▶️ Analyzing publication on topic {rcl_publish_topic}")

    # identify all subscriptions that are input to this publication
    rcl_subs = get_dependencies_of_publisher(rcl_publish_instance_end)
    
    for rcl_sub in rcl_subs:
        current_topic = rcl_subscription_to_topic_map[rcl_sub]
        print(f"     🔀 Following subscription on topic {current_topic}")
        
        rmw_take_instance = get_latest_taken_message_for_subscription(rcl_sub, rcl_publish_instance_end.timestamp)
        if rmw_take_instance is None:
            print(f"      ❌ rmw_take: No taken message found for subscribed topic {current_topic} at timestamp {timestamp_to_readable(rcl_publish_instance_end.timestamp)}")
            continue
        print(f"      ✅ rmw_take: Message taken at timestamp {timestamp_to_readable(rmw_take_instance.timestamp)}")

        if current_topic in begin_topics:
            print(f"    🏁 Reached begin topic {current_topic} at timestamp {timestamp_to_readable(rmw_take_instance.timestamp)}")
            # Update begin_time only if it's None or if we found an earlier time
            if time['begin_time'] is None or rmw_take_instance.timestamp < time['begin_time']:
                time['begin_time'] = rmw_take_instance.timestamp
                time['begin_topic_found'] = current_topic
            continue

        rmw_publish_instance = get_published_for_taken_message(rmw_take_instance)
        if rmw_publish_instance is None:
            print(f"      ❌ rmw_publish: No matching published message found for subscribed topic {current_topic} at timestamp {timestamp_to_readable(rmw_take_instance.timestamp)}")
            continue
        print(f"      ✅ rmw_publish: Message published at timestamp {timestamp_to_readable(rmw_publish_instance.timestamp)}")

        rcl_publish_instances = handler.data.rcl_publish_instances[
            (handler.data.rcl_publish_instances.message == rmw_publish_instance.message) &
            (handler.data.rcl_publish_instances.timestamp <= rmw_publish_instance.timestamp)
        ]
        rcl_publish_instance = rcl_publish_instances.loc[rcl_publish_instances.timestamp.idxmax()]
        if rcl_publish_instance.empty:
            raise ValueError(f"❌ rcl_publish: Could not identify single rcl_publish_instance for rmw_publish_instance 0x{rmw_publish_instance:x}")
        print(f"      ✅ rcl_publish: Message published at timestamp {timestamp_to_readable(rcl_publish_instance.timestamp)}")

        backpropagate_messages(rcl_publish_instance, begin_topics, time)


In [ ]:
critical_path_durations = {}
incomplete_paths = []

# get all publishers of end_topic
rcl_publishers_end_topic = handler.data.rcl_publishers[handler.data.rcl_publishers['topic_name'] == end_topic]
if len(rcl_publishers_end_topic) == 0:
    raise ValueError(f"No rcl_publishers found for topic '{end_topic}'")
elif len(rcl_publishers_end_topic) > 1:
    print(f"Warning: Multiple rcl_publishers found for topic '{end_topic}'. Will analyze all of them separately.")

for rcl_publisher, row in rcl_publishers_end_topic.iterrows():
    print(f"Analyzing critical path for end topic '{end_topic}' published by rcl_publisher '0x{rcl_publisher:x}'")

    # Find all published messages by this rcl_publisher
    rcl_publish_instances_end_topic = handler.data.rcl_publish_instances[handler.data.rcl_publish_instances['publisher_handle'] == rcl_publisher]
    print(f" Found {len(rcl_publish_instances_end_topic)} published messages for rcl_publisher '0x{rcl_publisher:x}'")
    num_analyzed = 0
    for pub_instance_idx, pub_instance_row in rcl_publish_instances_end_topic.iterrows():
        time = {'end_time': pub_instance_row.timestamp, 'begin_time': None, 'begin_topic_found': None}
        print(f"  {num_analyzed}: Analyzing rcl_publish_instance 0x{pub_instance_idx:x} at timestamp {timestamp_to_readable(time['end_time'])}")
        if num_analyzed == 61:
            pass
        transfer_durations_iteration = {}
        processing_durations_iteration = {}
        rcl_taken_begin = backpropagate_messages(pub_instance_row, begin_topic, time)
        if time['begin_time'] is None:
            print(f"   Could not find begin time for end time {timestamp_to_readable(time['end_time'])}")
            incomplete_paths.append(time['end_time'])
            continue
        duration = time['end_time'] - time['begin_time']
        print(f"  ✅ Critical path duration {duration/1e6} ms (from {time['begin_topic_found']})")
        critical_path_durations[time['end_time']] = duration
        num_analyzed += 1
        if max_num_analyze and num_analyzed >= max_num_analyze:
            print(f"Reached max_num_analyze limit of {max_num_analyze}. Stopping analysis for rcl_publisher '0x{rcl_publisher:x}'")
            break

## Compute Critical Path Durations

Convert the causal pairs (LiDAR callback → Ackermann publish) to duration measurements.

In [ ]:
if len(critical_path_durations) == 0:
    raise RuntimeError("No critical path flows found. The message links may not capture the causality.")

# Extract durations
timestamps_ns = np.fromiter(critical_path_durations.keys(), dtype=float)
durations_ms = np.array(list(critical_path_durations.values())) / 1e6

print(f"Could not follow {len(incomplete_paths)} incomplete paths: \n  - {', '.join([timestamp_to_readable(incomplete_path) for incomplete_path in incomplete_paths])}")
print(f"Computed {len(durations_ms)} critical-path durations")
print(f"Duration range: {durations_ms.min():.2f} ms to {durations_ms.max():.2f} ms")
print(f"Mean duration: {durations_ms.mean():.2f} ms")
print(f"Median duration: {np.median(durations_ms):.2f} ms")


## Analyze Message Flow

Plot histogram of durations and durations over end time

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(durations_ms, bins=50, color="#4C72B0", edgecolor="white")
ax.set_title(f"Critical Path Duration \nFrom:\n{',\n'.join(begin_topic)}\nTo:\n{end_topic}")
ax.set_xlabel("Duration (ms)")
ax.set_ylabel("Count")
ax.grid(True, alpha=0.2)

summary = pd.Series(durations_ms).describe(percentiles=[0.5, 0.9, 0.99])
print(summary)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator

# Convert timestamps from ms to datetime in Berlin timezone
timestamps_datetime = pd.to_datetime(timestamps_ns, unit='ns', utc=True)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(timestamps_datetime, durations_ms, marker='o', markersize=3, linestyle='-', linewidth=0.5, color="#4C72B0")
ax.set_title("Critical Path Duration Over Time")
ax.set_xlabel("End Time")
ax.set_ylabel("Duration (ms)")
ax.grid(True, alpha=0.2)

# Format x-axis with time including milliseconds
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S.%f', tz='UTC'))
# Increase the number of ticks
ax.xaxis.set_major_locator(MaxNLocator(nbins=20))
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()